<a href="https://colab.research.google.com/github/c-marq/cap4767-data-mining/blob/main/exercises/m04-05_group_exercise_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modules 4–5 Group Exercise, Part 1: Churn EDA + Logistic Regression
**CAP4767 Data Mining with Python** | Miami Dade College, Kendall Campus

**Points:** 5 (Part 1 of 2; the whole exercise is 10) | **Duration:** ~45 minutes | **Deliverable:** Completed notebook + 2–3 minute presentation

**Submission:** each student submits their own copy in Canvas. The work is done in groups, but every student turns in their own notebook (file name: `LastName_FirstName_Exercise04_Part1.ipynb`).

**Objective:** Perform statistical EDA on a bank churn dataset, estimate the business cost of churn, build a logistic regression baseline, and interpret coefficients as actionable business insights. The steps are the same as the Modules 4–5 demo, but the data is different.

### Your dataset

**Bank Customer Churn** (`Churn_Modelling.csv`): 10,000 customers of a retail bank in France, Germany and Spain. The target `Exited` is 1 if the customer closed their account and 0 if they stayed. About 20% of the customers left. Source: Kaggle, Churn Modelling (shrutimechlearn). This is new data, not the Telco data from the demo.

**Chapter 4 skills practiced:**
- Cramér's V (categorical association)
- Mann-Whitney U + Cohen's d (continuous feature analysis)
- Business cost estimation
- Logistic regression + coefficient interpretation

### Group Members & Roles

| Role | Name | Responsibility |
|------|------|----------------|
| 🖥️ **Lead Coder** | | Drives the notebook |
| 📊 **Data Interpreter** | | Reads outputs, explains metrics |
| 🎤 **Presenter** | | Delivers the 2–3 minute share-out |
| ✅ **QA Reviewer** | | Checks outputs against checkpoints |

*If 3 members, Lead Coder also handles QA.*

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 GROUP DISCUSSION (before coding, 3 minutes)</strong><br>
  You're a data team at a retail bank in France, Germany and Spain. Before looking at any data, discuss:
  <ol>
    <li>What features do you <em>think</em> predict whether a customer closes their account? List your top 3 guesses.</li>
    <li>How would you measure the cost of losing a customer? What numbers would you need?</li>
    <li>Why would a business prefer a model that can <em>explain</em> churn drivers over one that just predicts who will leave?</li>
  </ol>
</div>

**Our group's predictions and reasoning (minimum 3 sentences):**

*(Write here)*

---

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Run the setup cell below. It loads the Bank Customer Churn dataset and creates helper functions for your EDA. <strong>Do not modify.</strong>
</div>

In [ ]:
# ============================================================
# Setup. Run this cell. Do not modify.
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from scipy.stats import chi2_contingency, mannwhitneyu, pointbiserialr
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score)

np.random.seed(42)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

# Helper functions
def cramers_v(x, y):
    """Cramér's V: association between two categorical variables (0–1)."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

def cohens_d(group1, group2):
    """Cohen's d: effect size between two groups."""
    n1, n2 = len(group1), len(group2)
    pooled = np.sqrt(((n1-1)*group1.std()**2 + (n2-1)*group2.std()**2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled if pooled > 0 else 0

# Load and clean
url = "https://raw.githubusercontent.com/c-marq/cap4767-data-mining/refs/heads/main/data/Churn_Modelling.csv"
df = pd.read_csv(url)
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])  # row number, ID and name are not predictors

# Feature lists for your EDA
# NumOfProducts only takes the values 1 to 4, so it counts as a category here
cat_features = ["Geography", "Gender", "HasCrCard", "IsActiveMember", "NumOfProducts"]
num_features = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]

print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Churn rate (Exited = 1): {df['Exited'].mean():.1%}")
print(f"   Categorical features ({len(cat_features)}): {cat_features}")
print(f"   Numerical features ({len(num_features)}): {num_features}")
print(f"   Helper functions ready: cramers_v(), cohens_d()")

---
## Task 1: Cramér's V Analysis (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Compute Cramér's V between each feature in <code>cat_features</code> and <code>Exited</code>. Create a sorted horizontal bar chart.
</div>

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 WHY ARE WE DOING THIS?</strong><br>
  In Module 3, we used Pearson correlation for numeric features. Churn is a yes/no outcome (<code>Exited</code>: 1 = left, 0 = stayed), so we need Cramér's V instead. It works like correlation but for categories: 0 = no association, 1 = perfect association. <code>NumOfProducts</code> only holds the values 1 to 4, so it counts as a category.
</div>

In [ ]:
# Task 1: Cramér's V for all categorical features vs Exited
# Use the pre-built cramers_v() function and cat_features list
# YOUR CODE HERE


**Interpretation (2–3 sentences):** Which 2 features have the strongest association with churn? Were they in your pre-coding predictions?

*(Write here)*

---
## Task 2: Mann-Whitney U + Cohen's d (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  For each feature in <code>num_features</code>, run Mann-Whitney U and compute Cohen's d between churners and non-churners. Display as a table.
</div>

In [ ]:
# Task 2: Mann-Whitney U + Cohen's d
# Split: churn_yes = df[df["Exited"]==1], churn_no = df[df["Exited"]==0]
# For each num_feature: mannwhitneyu() + cohens_d()
# YOUR CODE HERE


**Interpretation (2–3 sentences):** Which numerical feature has the largest effect size? What does the sign of Cohen's d tell you about churners vs stayers?

*(Write here)*

<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 1</strong><br>
  <ul>
    <li><strong>Cramér's V:</strong> NumOfProducts should be first (about 0.39). Geography (about 0.17), IsActiveMember (about 0.16) and Gender (about 0.11) come next, and HasCrCard should be close to 0 (about 0.01)</li>
    <li><strong>Cohen's d:</strong> Age should show a medium positive effect (about +0.74; churners are older). Balance is small (about +0.30). CreditScore, Tenure and EstimatedSalary should all be close to 0 (under 0.07 in size)</li>
    <li><strong>Mann-Whitney U:</strong> Age and Balance have tiny p-values. Tenure (about 0.16) and EstimatedSalary (about 0.23) are above 0.05</li>
    <li>If HasCrCard or EstimatedSalary shows up as a strong driver in either test, double-check: both are typically negligible here</li>
  </ul>
</div>

---
## Task 3: Business Cost of Churn (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  Calculate the estimated annual cost of churn. Use these data points:
  <ul>
    <li><code>df.loc[df["Exited"]==1, "Balance"].sum()</code>: the total balance held by the customers who left</li>
    <li><code>df["Exited"].sum()</code>: number of churned customers</li>
    <li>Assume the bank earns 2% a year (0.02) on every dollar of balance a customer keeps with it</li>
    <li>Assume $300 acquisition cost per replacement customer (marketing, onboarding and a welcome offer)</li>
  </ul>
  Compute: (1) annual revenue at risk, (2) replacement cost, (3) total impact. State the two assumptions in your comments.
</div>

<div style="background-color: #D6EAF8; border-left: 5px solid #2E86C1; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1A5276;">💡 WHY ARE WE DOING THIS?</strong><br>
  Models don't convince leadership, dollars do. Before building any model, we establish what churn <em>costs</em>. This number frames every decision from here: "Our model missed 150 churners × $X per customer = $Y in preventable losses."
</div>

In [ ]:
# Task 3: Business cost of churn
# YOUR CODE HERE


---
## Task 4: Logistic Regression Baseline (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  The preprocessing cell below is pre-filled. Run it, then build a <code>LogisticRegression</code> and print the classification report. Store the fitted model as <code>lr_model</code>, the 0/1 predictions as <code>predictions</code> and the churn probabilities as <code>probabilities</code>, because Task 5 uses those names.
</div>

In [ ]:
# Pre-filled preprocessing. Run this cell. Do not modify.
df_model = df.copy()

df_model["Gender"] = df_model["Gender"].map({"Male": 1, "Female": 0})
# Geography and NumOfProducts become 0/1 dummy columns. NumOfProducts is a category on purpose:
# 2 products churn least and 3 or 4 churn most, which one straight-line coefficient cannot show.
df_model = pd.get_dummies(df_model, columns=["Geography", "NumOfProducts"],
                           drop_first=True, dtype=int)

X = df_model.drop(columns=["Exited"])
y = df_model["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
continuous = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
X_train[continuous] = scaler.fit_transform(X_train[continuous])
X_test[continuous] = scaler.transform(X_test[continuous])

feature_names = X_train.columns.tolist()
print(f"✅ Preprocessing complete: {len(feature_names)} features, {X_train.shape[0]:,} train / {X_test.shape[0]:,} test")

In [ ]:
# Task 4: Build logistic regression
# Use max_iter=1000, random_state=42
# Print classification report with target_names=["Stayed", "Churned"]
# YOUR CODE HERE
# Store: lr_model, predictions, probabilities


<div style="background-color: #FADBD8; border-left: 5px solid #E74C3C; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #922B21;">🛑 CHECKPOINT 2</strong><br>
  <ul>
    <li>Accuracy about 84% (anything from 82% to 86% is fine). Always predicting "Stayed" would already score about 80% on the test rows, so accuracy alone says little here</li>
    <li>Recall on "Churned" about 36% (30% to 45% is fine): the model catches a little over a third of actual churners. If recall comes out near 77%, check that you did not add <code>class_weight="balanced"</code></li>
    <li>AUC about 0.84</li>
  </ul>
</div>

---
## Task 5: Coefficient Interpretation + Confusion Matrix (1 pt)

<div style="background-color: #D5F5E3; border-left: 5px solid #27AE60; padding: 15px; margin: 15px 0; border-radius: 4px;">
  <strong style="color: #1E8449;">✅ DO THIS</strong><br>
  <ol>
    <li>Plot the top 5 positive and top 5 negative coefficients as a horizontal bar chart</li>
    <li>Generate and display the confusion matrix</li>
    <li>Calculate the dollar cost of the False Negatives (churners the model missed)</li>
  </ol>
  Reading the dummy columns: each one is compared with the dropped group (France for Geography, 1 product for NumOfProducts).
</div>

In [ ]:
# Task 5a: Coefficient bar chart, top 5 positive and top 5 negative
# YOUR CODE HERE


In [ ]:
# Task 5b: Confusion matrix + dollar cost of false negatives
# Hint: tn, fp, fn, tp = confusion_matrix(y_test, predictions).ravel()
# Cost per missed churner = average Balance of the churners x 0.02 (the same yearly revenue per churner as Task 3)
# YOUR CODE HERE


**Interpretation (minimum 4 sentences):** Explain the top 3 churn drivers to a non-technical executive. For each one, suggest a specific business action the bank could take.

*(Write here)*

---

## Share-Out (2–3 minutes)

Present to the class:
1. **Top 2 churn drivers** from Cramér's V and Cohen's d: did they match your predictions?
2. **Business cost:** what does churn cost this bank annually?
3. **Model performance:** how many churners does the model catch? How many slip through?
4. **One actionable recommendation** based on the coefficients

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `cramers_v()` gives error | Make sure you're passing two Series, not strings: `cramers_v(df["Geography"], df["Exited"])` |
| Cohen's d is 0 for all features | Check your group split: `churn_yes = df[df["Exited"]==1]`. Note the integer 1 (Exited holds 0 and 1), not the string "Yes" |
| Classification report shows all zeros for one class | Check that `y_train` holds both 0 and 1 and that you fit on `X_train`. (A missing `max_iter=1000` does not cause this: the default gives the same report.) |
| `NameError: name 'predictions' is not defined` (or `'lr_model'`) | Task 5 uses the names Task 4 stores: `lr_model`, `predictions`, `probabilities`. Rename your variables or re-run Task 4 with those names |
| `NameError: name 'y_test' is not defined` | Make sure you ran the preprocessing cell first |

---
<p style="color:#7F8C8D; font-size:0.85em;">
<em>CAP4767 Data Mining with Python | Miami Dade College</em><br>
Modules 4–5 Group Exercise, Part 1: Churn EDA + Logistic Regression | 5 Points
</p>